
# MARCS HDAC 再解析（Jupyter版）
このノートブックは、`results/tables/complexes_HDAC_key_features.csv` と `results/tables/proteins_key_features.csv` を読み込み、  
**(A) 複合体レベル**と **(B) タンパク質レベル**の両方で、**H3ac/H4ac/H3K4me3/H2A.Z** に対する**HDAC関連因子の相互作用**（効果量・有意性）を可視化する。

**前提**：
- Matplotlib のみ使用（Seaborn 不使用）。
- 図は**1セル＝1図**（サブプロット不使用）。
- 有意判定の既定：`FDR ≤ 0.01` かつ `|effect| ≥ 1`。
- 必要に応じて `marks` のリストを編集すれば、別マークも解析可能。


In [ ]:
# MARCS public path configuration
from pathlib import Path
import os

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analyses').is_dir() and (candidate / 'README.md').is_file():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', REPO_ROOT / 'data')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', REPO_ROOT / 'outputs')).expanduser().resolve()
MARCS_DATA_ROOT = GAO2026_DATA_ROOT / '13_MARCS'
MARCS_OUTPUT_ROOT = GAO2026_OUTPUT_ROOT / '13_MARCS'
MARCS_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# === パス設定（必要に応じて変更） ===
c_path = MARCS_DATA_ROOT / "complexes_HDAC_key_features.csv"
p_path = MARCS_DATA_ROOT / "proteins_key_features.csv"

assert c_path.exists(), f"Not found: {c_path}"
assert p_path.exists(), f"Not found: {p_path}"

# === パラメータ ===
marks = ["H3ac","H4ac","H3K4me3","H2A.Z"]   # 解析対象のマーク
alpha = 0.01                                 # FDR閾値
eff_thr = 1.0                                # |effect|の閾値


In [ ]:

# 取得
c_df = pd.read_csv(c_path)   # complexes
p_df = pd.read_csv(p_path)   # proteins

# 型整形
for df in (c_df, p_df):
    for col in ["effect","fdr"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# 欠損掃除（必要最小限）
c_df = c_df.dropna(subset=["effect","fdr","feature_std"])
p_df = p_df.dropna(subset=["effect","fdr","feature_std"])

# 有意フラグ
c_df["sig_flag"] = (c_df["fdr"] <= alpha) & (c_df["effect"].abs() >= eff_thr)
p_df["sig_flag"] = (p_df["fdr"] <= alpha) & (p_df["effect"].abs() >= eff_thr)

# クリーンなラベル（protein側）
if "label_for_plot" in p_df.columns:
    p_df["gene_label"] = p_df["label_for_plot"].astype(str)
else:
    p_df["gene_label"] = p_df["gene"].astype(str)

print("Complex rows:", len(c_df), "complexes:", c_df["complex_name"].nunique())
print("Protein rows:", len(p_df), "genes:", p_df["gene_label"].nunique())


## (A) 複合体レベル：Complex × Mark の効果量ヒートマップ（有意セルに★）

In [ ]:

# ピボット（行＝complex_std、列＝feature_std、値＝effect）
cx_pivot = (c_df
    .loc[c_df["feature_std"].isin(marks)]
    .pivot_table(index="complex_std", columns="feature_std", values="effect", aggfunc="median")
    .reindex(index=sorted(c_df["complex_std"].dropna().unique()))
    .reindex(columns=marks)
)

# 有意セルのマスク
sig_pivot = (c_df
    .assign(sig_numeric=lambda d: d["sig_flag"].astype(int))
    .loc[c_df["feature_std"].isin(marks)]
    .pivot_table(index="complex_std", columns="feature_std", values="sig_numeric", aggfunc="max")
    .reindex(index=cx_pivot.index, columns=cx_pivot.columns)
).fillna(0).astype(int)

# 図：imshow + ★注記
fig = plt.figure(figsize=(6, 0.6*len(cx_pivot.index) + 2))
ax = plt.gca()
im = ax.imshow(cx_pivot.values, aspect="auto")  # カラーマップは指定しない（既定使用）
ax.set_xticks(range(len(cx_pivot.columns)))
ax.set_xticklabels(cx_pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(cx_pivot.index)))
ax.set_yticklabels(cx_pivot.index)
ax.set_title("Complex-level effects (median)")
plt.colorbar(im, ax=ax, label="effect")

# 有意セルに★
for i, row in enumerate(cx_pivot.index):
    for j, col in enumerate(cx_pivot.columns):
        if sig_pivot.iat[i, j] == 1:
            ax.text(j, i, "★", ha="center", va="center")

plt.tight_layout()
plt.show()


### 複合体：各マークの効果 vs 有意度（-log10 FDR）散布図（1図/マーク）

In [ ]:

import numpy as np

for mk in marks:
    sub = c_df[(c_df["feature_std"]==mk)]
    if sub.empty:
        continue
    x = sub["effect"].values
    y = -np.log10(sub["fdr"].clip(lower=1e-300)).values
    labels = sub["complex_std"].fillna(sub["complex_name"]).astype(str).values

    fig = plt.figure(figsize=(5,4))
    ax = plt.gca()
    ax.scatter(x, y, s=30)
    ax.axvline(0, ls="--", lw=1)
    ax.set_xlabel("effect (Δ log2[H/L])")
    ax.set_ylabel("-log10(FDR)")
    ax.set_title(f"Complexes — {mk}")
    # 上位をラベル（上位8件）
    idx = np.argsort(-y)[:8]
    for i in idx:
        ax.text(x[i], y[i], labels[i], fontsize=8)
    plt.tight_layout()
    plt.show()


## (B) タンパク質レベル：HDAC関連サブユニットのみ抽出して可視化

In [ ]:

# 代表的なHDAC関連サブユニット集合（必要に応じて編集）
panel = {
    "NuRD": ["HDAC1","HDAC2","CHD3","CHD4","MTA1","MTA2","MTA3","GATAD2A","GATAD2B","MBD2","MBD3","RBBP4","RBBP7"],
    "Sin3": ["HDAC1","HDAC2","SIN3A","SIN3B","SAP30","SAP30BP","SUDS3","ARID4A","ARID4B","RBBP4","RBBP7"],
    "CoREST": ["HDAC1","HDAC2","KDM1A","RCOR1","RCOR2","RCOR3","PHF21A"],
    "MiDAC": ["HDAC1","HDAC2","ELMSAN1","DNTTIP1"],
    "NCoR/SMRT": ["HDAC3","NCOR1","NCOR2","TBL1X","TBL1XR1","GPS2"],
}

# 大文字で突き合わせ
p_df["gene_upper"] = p_df["gene_label"].str.upper()

panel_all = sorted(set(g for genes in panel.values() for g in genes))
p_sub = p_df[p_df["gene_upper"].isin(panel_all) & p_df["feature_std"].isin(marks)].copy()

# マーク×遺伝子行列（効果量）
mat = (p_sub
       .pivot_table(index="gene_label", columns="feature_std", values="effect", aggfunc="median")
       .reindex(columns=marks))

# 有意セル
sig_mat = (p_sub
    .assign(sig_numeric=lambda d: d["sig_flag"].astype(int))
    .pivot_table(index="gene_label", columns="feature_std", values="sig_numeric", aggfunc="max")
    .reindex(index=mat.index, columns=mat.columns)
).fillna(0).astype(int)

print("HDAC-related proteins found:", len(mat.index))
mat.head()


In [ ]:

# 長くなる場合は上位Nだけにするには、コメントアウトを外す
# N = 40
# mat = mat.iloc[:N]
# sig_mat = sig_mat.loc[mat.index]

fig = plt.figure(figsize=(6, 0.25*len(mat.index) + 2))
ax = plt.gca()
im = ax.imshow(mat.values, aspect="auto")  # 既定カラーマップ
ax.set_xticks(range(len(mat.columns)))
ax.set_xticklabels(mat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(mat.index)))
ax.set_yticklabels(mat.index)
ax.set_title("Proteins (HDAC-related) — effect matrix")
plt.colorbar(im, ax=ax, label="effect")

# 有意セルに★
for i, row in enumerate(mat.index):
    for j, col in enumerate(mat.columns):
        if sig_mat.iat[i, j] == 1:
            ax.text(j, i, "★", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:

for mk in marks:
    sub = p_sub[p_sub["feature_std"]==mk].copy()
    if sub.empty:
        continue
    x = sub["effect"].values
    y = -np.log10(sub["fdr"].clip(lower=1e-300)).values
    labels = sub["gene_label"].astype(str).values

    fig = plt.figure(figsize=(5,4))
    ax = plt.gca()
    ax.scatter(x, y, s=20)
    ax.axvline(0, ls="--", lw=1)
    ax.set_xlabel("effect (Δ log2[H/L])")
    ax.set_ylabel("-log10(FDR)")
    ax.set_title(f"HDAC-related proteins — {mk}")
    # 上位をラベル（上位12件）
    idx = np.argsort(-y)[:12]
    for i in idx:
        ax.text(x[i], y[i], labels[i], fontsize=8)
    plt.tight_layout()
    plt.show()


### 集計（有意件数・方向性など）

In [ ]:

# 複合体レベル：マークごとの有意件数（閾値は alpha, eff_thr）
cx_sum = (c_df[c_df["feature_std"].isin(marks)]
          .assign(dir=lambda d: np.where(d["effect"]>0,"up",np.where(d["effect"]<0,"down","zero")))
          .groupby("feature_std")
          .apply(lambda d: pd.Series({
              "n_sig": int(((d["fdr"]<=alpha) & (d["effect"].abs()>=eff_thr)).sum()),
              "n_up_sig": int(((d["fdr"]<=alpha) & (d["effect"]>=eff_thr)).sum()),
              "n_down_sig": int(((d["fdr"]<=alpha) & (d["effect"]<=-eff_thr)).sum()),
              "median_effect": float(d["effect"].median())
          }))
         )

# タンパク質レベル：HDACパネルのみ
prot_sum = (p_sub
            .assign(dir=lambda d: np.where(d["effect"]>0,"up",np.where(d["effect"]<0,"down","zero")))
            .groupby("feature_std")
            .apply(lambda d: pd.Series({
                "n": int(d.shape[0]),
                "n_sig": int(((d["fdr"]<=alpha) & (d["effect"].abs()>=eff_thr)).sum()),
                "n_up_sig": int(((d["fdr"]<=alpha) & (d["effect"]>=eff_thr)).sum()),
                "n_down_sig": int(((d["fdr"]<=alpha) & (d["effect"]<=-eff_thr)).sum()),
                "median_effect": float(d["effect"].median())
            }))
           )

print("=== Complex-level summary ===")
display(cx_sum)

print("\n=== Protein-level (HDAC panel) summary ===")
display(prot_sum)


以下は **`marcs.ptm-response-complexes.2022-03-22.824b7d4.xlsx`** の**全タブ（Index of complexes を除く）**に対し、

* **縦軸：−log10(FDR)**（`Adjusted P value (BH)` を使用。無い場合は `P value` からBH補正で代替）
* **横軸：effect**（`Median effect` を優先、無ければ effect 系の列を自動検出）
* **「Proteins for which response was measurable」に “HDAC” を含む行**を**上書き散布（marker='x'）**し、**Complex名を注記**
* 参照線：x=0、x=±1、y=−log10(0.01)
  を描く**ボルケーノ図をタブごとに1図ずつ**作成・保存する **Jupyter Notebook 用セル**である。
  （作図は**Matplotlibのみ**、**サブプロット不使用**、**色は指定しない**＝既定色のまま、という制約を満たしている。）

> **貼り付け場所**：前回お渡しした Notebook の続きの新セルとして、そのまま実行すればよい。

---

### セル1：ユーティリティとパス設定、全タブ処理＋作図（保存＆表示）


In [ ]:
# === Volcano plots for all tabs in the complexes workbook with HDAC highlight ===
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 入力ファイル（必要に応じてパスを調整）
cmp_path = MARCS_DATA_ROOT / "marcs.ptm-response-complexes.2022-03-22.824b7d4.xlsx"
# 出力先
outdir = MARCS_OUTPUT_ROOT / "figures/complexes_volcano_alltabs"
outdir.mkdir(parents=True, exist_ok=True)

def _norm(s):
    return "".join(ch for ch in str(s).strip().lower() if ch.isalnum())

def _find_col(cols, candidates):
    """列名の表記ゆれに耐性を持たせた検索（最初に完全一致→部分一致）。"""
    norm_map = {_norm(c): c for c in cols}
    # 完全一致優先
    for cand in candidates:
        key = _norm(cand)
        if key in norm_map:
            return norm_map[key]
    # 部分一致も許容
    for c in cols:
        nc = _norm(c)
        if any(_norm(cand) in nc for cand in candidates):
            return c
    return None

def _safe_neglog10(x):
    x = pd.to_numeric(x, errors="coerce")
    x = x.clip(lower=1e-300)
    return -np.log10(x)

# Excel の全シートを走査（Index of complexes は除外）
xl = pd.ExcelFile(cmp_path)
sheets = [s for s in xl.sheet_names if not s.lower().startswith("index")]

for sheet in sheets:
    df = xl.parse(sheet, header=0)

    # 列名を検出
    cx_col   = _find_col(df.columns, ["Complex", "complex", "complex name"])
    eff_col  = _find_col(df.columns, ["Median effect", "Effect", "Average effect per pull-down pair"])
    fdr_col  = _find_col(df.columns, [
        "Adjusted P value (BH)", "BH adjusted p-value", "Adjusted p-value",
        "adjusted p value (bh)", "FDR", "q_value", "adj_p", "adj_p_value"
    ])
    p_col    = _find_col(df.columns, ["P value", "p-value", "p"])
    prot_col = _find_col(df.columns, ["Proteins for which response was measurable"])

    # 最低限の列チェック
    if cx_col is None or eff_col is None:
        print(f"[warn] {sheet}: 'Complex' もしくは 'effect' 列が見つからないためスキップ")
        continue

    # FDR を用意（FDR列が無い場合は P からBH補正で代替）
    if fdr_col is None:
        if p_col is None:
            print(f"[warn] {sheet}: FDR / P value 列が見つからないためスキップ")
            continue
        vals = pd.to_numeric(df[p_col], errors="coerce")
        from statsmodels.stats.multitest import multipletests
        fdr = pd.Series(np.nan, index=df.index)
        mask = vals.notna()
        if mask.any():
            fdr.loc[mask] = multipletests(vals[mask].values, method="fdr_bh")[1]
        df["_FDR_auto_"] = fdr
        fdr_col = "_FDR_auto_"

    # x, y のベクトル作成
    x = pd.to_numeric(df[eff_col], errors="coerce")
    y = _safe_neglog10(df[fdr_col])

    # HDAC を含む行のマスク（Proteins for which response was measurable）
    if prot_col is not None:
        hdac_mask = df[prot_col].astype(str).str.contains("HDAC", case=True, na=False)
    else:
        hdac_mask = pd.Series(False, index=df.index)

    # 図（各シート1図）
    fig = plt.figure(figsize=(6, 4))
    ax = plt.gca()

    # 全点
    ax.scatter(x, y, s=24)

    # HDAC含有の点を上書き（markerのみ変える；色は既定）
    if hdac_mask.any():
        ax.scatter(x[hdac_mask], y[hdac_mask], s=50, marker='x', label='contains "HDAC"')
        # Complex名を注記
        labels = df.loc[hdac_mask, cx_col].astype(str)
        for xi, yi, lab in zip(x[hdac_mask], y[hdac_mask], labels):
            # 軽微にずらして重なりを緩和（色指定なし）
            ax.text(xi, yi, lab, fontsize=8)

    # 参照線（色は既定）
    ax.axvline(0,  ls="--", lw=1)
    ax.axvline(1,  ls=":",  lw=1)
    ax.axvline(-1, ls=":",  lw=1)
    ax.axhline(-np.log10(0.01), ls=":", lw=1)

    ax.set_xlabel("effect (median)")
    ax.set_ylabel("-log10(FDR)")
    ax.set_title(f"Complex volcano — {sheet}")
    if hdac_mask.any():
        ax.legend(loc="best", frameon=False)

    plt.tight_layout()
    plt.savefig(outdir / f"volcano_complexes_{sheet}_HDAC.png", dpi=200)
    plt.show()


* **出力**：`results/figures/complexes_volcano_alltabs/volcano_complexes_{タブ名}_HDAC.png` が各タブごとに保存される。
* **凡例**：HDAC含有の点は `marker='x'` で上書きし、**Complex名を注記**している。
* **注記数が多すぎる場合**は、`for xi, yi, lab in ...` の前に上位選抜（例：`yが大きい順に上位N件`）のロジックを入れるとよい。

---

### セル2（任意）：HDAC該当点のサマリー表（タブ横断）

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

cmp_path = MARCS_DATA_ROOT / "marcs.ptm-response-complexes.2022-03-22.824b7d4.xlsx"
xl = pd.ExcelFile(cmp_path)
sheets = [s for s in xl.sheet_names if not s.lower().startswith("index")]

rows = []
for sheet in sheets:
    df = xl.parse(sheet, header=0)
    # 列検出（上セルの関数を再利用してもよい）
    def _norm(s): return "".join(ch for ch in str(s).strip().lower() if ch.isalnum())
    def _find(cols, cands):
        m = {_norm(c): c for c in cols}
        for z in cands:
            if _norm(z) in m: return m[_norm(z)]
        for c in cols:
            if any(_norm(z) in _norm(c) for z in cands): return c
        return None

    cx_col   = _find(df.columns, ["Complex", "complex", "complex name"])
    eff_col  = _find(df.columns, ["Median effect", "Effect", "Average effect per pull-down pair"])
    fdr_col  = _find(df.columns, ["Adjusted P value (BH)", "BH adjusted p-value", "Adjusted p-value", "FDR", "q_value", "adj_p", "adj_p_value"])
    p_col    = _find(df.columns, ["P value", "p-value", "p"])
    prot_col = _find(df.columns, ["Proteins for which response was measurable"])
    if cx_col is None or eff_col is None: 
        continue

    if fdr_col is None and p_col is not None:
        from statsmodels.stats.multitest import multipletests
        vals = pd.to_numeric(df[p_col], errors="coerce")
        fdr = pd.Series(np.nan, index=df.index)
        mask = vals.notna()
        if mask.any():
            fdr.loc[mask] = multipletests(vals[mask].values, method="fdr_bh")[1]
        df["_FDR_auto_"] = fdr
        fdr_col = "_FDR_auto_"

    # HDAC含有のみ抽出
    if prot_col is None:
        continue
    mask = df[prot_col].astype(str).str.contains("HDAC", case=True, na=False)
    sub = df.loc[mask, [cx_col, eff_col, fdr_col]].copy()
    sub.columns = ["complex_name", "effect", "fdr"]
    sub["sheet"] = sheet
    rows.append(sub)

hdac_list = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["complex_name","effect","fdr","sheet"])
hdac_list.sort_values(["sheet","complex_name"]).reset_index(drop=True)


了解した。ご提示のセルを、**対象タブを H3ac / H4ac / H3K4me3 に限定**し、**閾値を `effect=±0.5`・`-log10(FDR)=2`**、**軸範囲を x=[-2.5, 2.5], y=[0, 11]** に固定し、**HDAC含有（“Proteins for which response was measurable” に “HDAC” を含む）行を強調・Complex名を注記**するように改修した。
Matplotlibのみ・1図/セル・色指定なし（既定色）という制約を満たす。

以下の **1セル** をノートブック末尾に追加して実行されたし（パスは環境に合わせて調整）。

In [ ]:
# === Volcano plots for H3ac/H4ac/H3K4me3 only; HDAC highlight; fixed axes & thresholds
# === ラベル重なり回避＋leader line 付き注記 ===

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 入力・出力
cmp_path = MARCS_DATA_ROOT / "marcs.ptm-response-complexes.2022-03-22.824b7d4.xlsx"
outdir = MARCS_OUTPUT_ROOT / "figures/complexes_volcano_selected"
outdir.mkdir(parents=True, exist_ok=True)

# 対象タブ
marks = ["H3ac", "H4ac", "H3K4me3"]

# 閾値・軸範囲
x_thr_low, x_thr_high = -0.5, 0.5
y_thr = 2.0
x_min, x_max = -2.5, 2.5
y_min, y_max = 0.0, 11.0

def _norm(s):
    return "".join(ch for ch in str(s).strip().lower() if ch.isalnum())

def _find_col(cols, candidates):
    norm_map = {_norm(c): c for c in cols}
    for cand in candidates:
        key = _norm(cand)
        if key in norm_map:
            return norm_map[key]
    for c in cols:
        nc = _norm(c)
        if any(_norm(cand) in nc for cand in candidates):
            return c
    return None

def _safe_neglog10(v):
    v = pd.to_numeric(v, errors="coerce").clip(lower=1e-300)
    return -np.log10(v)

def annotate_with_leaders(ax, xs, ys, labels, 
                          max_labels=None,
                          base_offsets=((12, 6), (12, -6), (-12, 6), (-12, -6),
                                        (16, 0), (-16, 0), (0, 10), (0, -10)),
                          step_points=6,  # ラベルを遠ざけるときの増分（ピクセル）
                          max_radius=36   # これ以上離さない（ピクセル）
                          ):
    """
    点の近傍（ピクセル単位のオフセット）にラベルを配置し、既存ラベルとかぶる場合のみ
    オフセット半径を少しずつ増やして再配置する。各ラベルには leader line を描く。
    色指定はしない（デフォルト色）。
    """
    fig = ax.figure
    fig.canvas.draw()  # renderer を確定
    renderer = fig.canvas.get_renderer()
    placed_bboxes = []

    # 優先度：y（-log10FDR）が大きい順にラベル
    order = np.argsort(-ys.to_numpy())
    if max_labels is not None:
        order = order[:max_labels]

    for idx in order:
        xi, yi, lab = xs.iloc[idx], ys.iloc[idx], str(labels.iloc[idx])
        if pd.isna(xi) or pd.isna(yi) or not lab:
            continue

        # 半径を広げながら候補オフセットを試す
        radius = 0
        ann = None
        placed = False
        while radius <= max_radius and not placed:
            for dx0, dy0 in base_offsets:
                dx = dx0 + np.sign(dx0) * radius
                dy = dy0 + np.sign(dy0) * radius
                ha = "left" if dx >= 0 else "right"
                va = "bottom" if dy >= 0 else "top"

                # 一旦描いて bbox を確認
                ann = ax.annotate(
                    lab,
                    xy=(xi, yi),
                    xytext=(dx, dy),
                    textcoords="offset points",
                    ha=ha, va=va,
                    arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=0, 
                                    connectionstyle="arc3"),
                    zorder=5
                )
                fig.canvas.draw()
                bb = ann.get_window_extent(renderer=renderer).expanded(1.05, 1.2)
                # 既存ラベルと衝突？
                if any(bb.overlaps(b) for b in placed_bboxes):
                    ann.remove()
                    ann = None
                    continue
                # 衝突しなければ採用
                placed_bboxes.append(bb)
                placed = True
                break
            radius += step_points  # だめなら半径を少しだけ拡げる

        # どの候補でも置けなかった時：最初の位置で妥協
        if not placed and ann is None:
            ann = ax.annotate(
                lab, xy=(xi, yi), xytext=base_offsets[0], textcoords="offset points",
                ha="left", va="bottom",
                arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=0, connectionstyle="arc3"),
                zorder=5
            )
            fig.canvas.draw()
            placed_bboxes.append(ann.get_window_extent(renderer=renderer))

xl = pd.ExcelFile(cmp_path)
sheets = [s for s in marks if s in xl.sheet_names]

for sheet in sheets:
    df = xl.parse(sheet, header=0)

    cx_col   = _find_col(df.columns, ["Complex", "complex", "complex name"])
    eff_col  = _find_col(df.columns, ["Median effect", "Effect", "Average effect per pull-down pair"])
    fdr_col  = _find_col(df.columns, ["Adjusted P value (BH)", "BH adjusted p-value",
                                      "Adjusted p-value", "adjusted p value (bh)", "FDR",
                                      "q_value", "adj_p", "adj_p_value"])
    p_col    = _find_col(df.columns, ["P value", "p-value", "p"])
    prot_col = _find_col(df.columns, ["Proteins for which response was measurable"])

    if cx_col is None or eff_col is None:
        print(f"[warn] {sheet}: 'Complex' もしくは 'effect' 列が見つからないためスキップ")
        continue

    # FDR列が無い場合は P からBH補正
    if fdr_col is None:
        if p_col is None:
            print(f"[warn] {sheet}: FDR / P value 列が見つからないためスキップ")
            continue
        from statsmodels.stats.multitest import multipletests
        p = pd.to_numeric(df[p_col], errors="coerce")
        fdr = pd.Series(np.nan, index=df.index)
        mask = p.notna()
        if mask.any():
            fdr.loc[mask] = multipletests(p[mask].values, method="fdr_bh")[1]
        df["_FDR_auto_"] = fdr
        fdr_col = "_FDR_auto_"

    # 座標
    x = pd.to_numeric(df[eff_col], errors="coerce")
    y = _safe_neglog10(df[fdr_col])

    # HDAC含有行（注記対象）
    if prot_col is not None:
        hdac_mask = df[prot_col].astype(str).str.contains("HDAC", case=False, na=False)
    else:
        hdac_mask = pd.Series(False, index=df.index)

    # カテゴリ
    up_strong   = (x >=  x_thr_high) & (y >= y_thr)
    down_strong = (x <= x_thr_low)   & (y >= y_thr)
    mid_sig     = (x > x_thr_low) & (x < x_thr_high) & (y >= y_thr)
    nonsig      = ~(up_strong | down_strong | mid_sig)

    # 描画
    fig = plt.figure(figsize=(6, 4))
    ax = plt.gca()

    # 非有意（薄め・小さめ）
    ax.scatter(x[nonsig], y[nonsig], s=12, alpha=0.3, zorder=1)
    # 有意：中心域
    ax.scatter(x[mid_sig], y[mid_sig], s=18, alpha=0.7, zorder=2, label="FDR≤0.01 & |effect|<0.5")
    # 有意：左右
    ax.scatter(x[up_strong], y[up_strong], s=28, zorder=3, label="Up (≥0.5, FDR≤0.01)")
    ax.scatter(x[down_strong], y[down_strong], s=28, zorder=3, label="Down (≤-0.5, FDR≤0.01)")

    # HDAC点（×マーカー）＋ leader line 付き注記
    if hdac_mask.any():
        ax.scatter(x[hdac_mask], y[hdac_mask], s=60, marker='x', zorder=4, label='contains "HDAC"')
        annotate_with_leaders(ax,
                              x[hdac_mask].reset_index(drop=True),
                              y[hdac_mask].reset_index(drop=True),
                              df.loc[hdac_mask, cx_col].astype(str).reset_index(drop=True),
                              max_labels=None)  # 上位Nだけにするなら整数を指定

    # 閾値線
    ax.axvline(0,           ls="--", lw=1)
    ax.axvline(x_thr_high,  ls=":",  lw=1)
    ax.axvline(x_thr_low,   ls=":",  lw=1)
    ax.axhline(y_thr,       ls=":",  lw=1)

    # 軸・凡例
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel(f"{sheet} effect to log H/L ratio (median)")
    ax.set_ylabel("-log10(FDR)")
    ax.set_title(f"Complex volcano — {sheet}")
    ax.legend(loc="upper left", frameon=False)

    plt.tight_layout()
    plt.savefig(outdir / f"volcano_complexes_{sheet}_HDAC_leaders.png", dpi=200)
    plt.show()


In [ ]:
# === Volcano plots (H3ac/H4ac/H3K4me3) — per-sheet recompute & fixed layout ===
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
from statsmodels.stats.multitest import multipletests
import traceback

# フォント設定（任意）
if "Arial" in {f.name for f in font_manager.fontManager.ttflist}:
    rcParams["font.family"] = "Arial"

cmp_path = MARCS_DATA_ROOT / "marcs.ptm-response-complexes.2022-03-22.824b7d4.xlsx"
outdir = MARCS_OUTPUT_ROOT / "figures/complexes_volcano_selected"
outdir.mkdir(parents=True, exist_ok=True)

marks = ["H3ac", "H4ac", "H3K4me3"]

# 閾値/軸
x_thr_low, x_thr_high = -0.5, 0.5
y_thr = 2.0
x_min, x_max = -2.5, 2.5
y_min, y_max = 0.0, 11.0

def _norm(s): return "".join(ch for ch in str(s).strip().lower() if ch.isalnum())
def _find_col(cols, candidates):
    m = {_norm(c): c for c in cols}
    for cand in candidates:
        key = _norm(cand)
        if key in m: return m[key]
    for c in cols:
        if any(_norm(cand) in _norm(c) for cand in candidates):
            return c
    return None

def _safe_neglog10(v):
    v = pd.to_numeric(v, errors="coerce").clip(lower=1e-300)
    return -np.log10(v)

def annotate_with_leaders(ax, xs, ys, labels,
                          max_labels=None,
                          base_offsets=((12, 6),(12,-6),(-12,6),(-12,-6),(16,0),(-16,0),(0,10),(0,-10)),
                          step_points=6, max_radius=36):
    fig = ax.figure; fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    placed = []
    order = np.argsort(-ys.to_numpy())
    if max_labels is not None: order = order[:max_labels]
    for idx in order:
        xi, yi, lab = xs.iloc[idx], ys.iloc[idx], str(labels.iloc[idx])
        if pd.isna(xi) or pd.isna(yi) or not lab: continue
        radius = 0; ann = None; ok = False
        while radius <= max_radius and not ok:
            for dx0, dy0 in base_offsets:
                dx = dx0 + np.sign(dx0)*radius
                dy = dy0 + np.sign(dy0)*radius
                ha = "left" if dx>=0 else "right"
                va = "bottom" if dy>=0 else "top"
                ann = ax.annotate(lab, xy=(xi,yi), xytext=(dx,dy),
                                  textcoords="offset points", ha=ha, va=va,
                                  arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=0,
                                                  connectionstyle="arc3"),
                                  zorder=5, clip_on=False)  # ★
                fig.canvas.draw()
                bb = ann.get_window_extent(renderer=renderer).expanded(1.05, 1.2)
                if any(bb.overlaps(b) for b in placed):
                    ann.remove(); ann=None; continue
                placed.append(bb); ok=True; break
            radius += step_points
        if not ok and ann is None:
            ann = ax.annotate(lab, xy=(xi,yi), xytext=base_offsets[0],
                              textcoords="offset points", ha="left", va="bottom",
                              arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=0,
                                              connectionstyle="arc3"),
                              zorder=5, clip_on=False)
            fig.canvas.draw(); placed.append(ann.get_window_extent(renderer=renderer))

# === ここから per-sheet ===
xl = pd.ExcelFile(cmp_path)
sheet_map = {s.lower(): s for s in xl.sheet_names}
sheets = [sheet_map[m.lower()] for m in marks if m.lower() in sheet_map]

for sheet in sheets:
    try:
        df = xl.parse(sheet, header=0)

        # 列検出（毎回）
        cx_col   = _find_col(df.columns, ["Complex","complex","complex name"])
        eff_col  = _find_col(df.columns, ["Median effect","Effect","Average effect per pull-down pair"])
        fdr_col  = _find_col(df.columns, ["Adjusted P value (BH)","BH adjusted p-value",
                                          "Adjusted p-value","adjusted p value (bh)","FDR","q_value","adj_p","adj_p_value"])
        p_col    = _find_col(df.columns, ["P value","p-value","p"])
        prot_col = _find_col(df.columns, ["Proteins for which response was measurable"])

        if cx_col is None or eff_col is None:
            print(f"[warn] {sheet}: missing required columns -> skip"); continue

        # FDR再計算の必要があれば毎回行う
        if fdr_col is None:
            if p_col is None:
                print(f"[warn] {sheet}: no FDR/P -> skip"); continue
            pvals = pd.to_numeric(df[p_col], errors="coerce")
            fdr = pd.Series(np.nan, index=df.index)
            mask = pvals.notna()
            if mask.any():
                fdr.loc[mask] = multipletests(pvals[mask].values, method="fdr_bh")[1]
            df["_FDR_auto_"] = fdr
            fdr_col = "_FDR_auto_"

        # ★ ここが肝：各シートで x, y, hdac_mask を作り直す
        x = pd.to_numeric(df[eff_col], errors="coerce")
        y = _safe_neglog10(df[fdr_col])
        if prot_col is not None:
            hdac_mask = df[prot_col].astype(str).str.contains("HDAC", case=False, na=False)
        else:
            hdac_mask = pd.Series(False, index=df.index)

        up_strong   = (x >=  x_thr_high) & (y >= y_thr)
        down_strong = (x <= x_thr_low)   & (y >= y_thr)
        mid_sig     = (x > x_thr_low) & (x < x_thr_high) & (y >= y_thr)
        nonsig      = ~(up_strong | down_strong | mid_sig)

        # 図を全て同じレイアウトに固定
        fig = plt.figure(figsize=(6.0, 4.0))
        ax = plt.gca()
        plt.subplots_adjust(left=0.12, right=0.98, bottom=0.16, top=0.88)

        ax.scatter(x[nonsig], y[nonsig], s=12, alpha=0.3, zorder=1)
        ax.scatter(x[mid_sig], y[mid_sig], s=18, alpha=0.7, zorder=2, label="FDR≤0.01 & |effect|<0.5")
        ax.scatter(x[up_strong], y[up_strong], s=28, zorder=3, label="Up (≥0.5, FDR≤0.01)")
        ax.scatter(x[down_strong], y[down_strong], s=28, zorder=3, label="Down (≤-0.5, FDR≤0.01)")

        if hdac_mask.any():
            ax.scatter(x[hdac_mask], y[hdac_mask], s=60, marker='x', zorder=4, label='contains "HDAC"')
            annotate_with_leaders(ax,
                                  x[hdac_mask].reset_index(drop=True),
                                  y[hdac_mask].reset_index(drop=True),
                                  df.loc[hdac_mask, cx_col].astype(str).reset_index(drop=True),
                                  max_labels=None)

        ax.axvline(0,          ls="--", lw=1)
        ax.axvline(x_thr_high, ls=":",  lw=1)
        ax.axvline(x_thr_low,  ls=":",  lw=1)
        ax.axhline(y_thr,      ls=":",  lw=1)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel(f"{sheet} effect to log H/L ratio (median)")
        ax.set_ylabel("-log10(FDR)")
        ax.set_title(f"Complex volcano — {sheet}")
        ax.legend(loc="upper left", frameon=False)

        base = f"volcano_complexes_{sheet}_HDAC_leaders"
        fig.savefig(outdir / f"{base}.png", dpi=200)   # bbox_inches は使わない
        fig.savefig(outdir / f"{base}.pdf")
        plt.close(fig)  # ★ クリアして次の反復へ

    except Exception as e:
        print(f"[error] Failed on sheet {sheet}: {e}")
        traceback.print_exc()


**Proteinsワークブック**（`marcs.ptm-response-proteins.2022-03-22.824b7d4.xlsx`）から **HDAC1/HDAC2/HDAC3** の効果推定を**全タブ横断**で集計し、添付図のような「**各マーク（シート）をY軸に並べ、X軸に効果量（log2）**、**中央値●＋IQR帯（Q1–Q3の太線）＋min–maxの細線**、さらに**各リプリケートの点**を重ねる」図を作る **Jupyter用コード**を提示する。
HDAC1/2/3は**色分け**し、同一マーク内で**わずかに縦方向オフセット**して重なりを避ける。ヘッダーが `Metadata` 行＋次行が実ヘッダーという **本ワークブック特有の構造**へも自動対応する。

> そのままノートブックの新セルに貼り付けて実行されたし（パスは必要に応じて修正）。Matplotlibのみ使用。

---

## セル1：読み込み・集計（HDAC1/2/3の効果分布を作る）

In [ ]:
# === HDAC1/2/3: Proteins workbook から全タブ横断で効果分布を抽出・集計 ===
from pathlib import Path
import pandas as pd
import numpy as np

wb_path = MARCS_DATA_ROOT / "marcs.ptm-response-proteins.2022-03-22.824b7d4.xlsx"  # 必要なら修正
genes_of_interest = ["HDAC1", "HDAC2", "HDAC3"]  # 色分け対象

# ---- ヘッダー検出ユーティリティ（proteins特有: 最初がMetadata＋次行が見出し） ----
def _norm(s):
    return "".join(ch for ch in str(s).strip().lower() if ch.isalnum())

def _find_col(cols, candidates):
    norm_map = {_norm(c): c for c in cols}
    # 完全一致優先
    for cand in candidates:
        key = _norm(cand)
        if key in norm_map:
            return norm_map[key]
    # 部分一致許容
    for c in cols:
        nc = _norm(c)
        if any(_norm(cand) in nc for cand in candidates):
            return c
    return None

def read_protein_sheet(xl, sheet):
    """Metadata行→次行が実ヘッダーの形式に対応。該当しない場合は通常のheader=0。"""
    raw = xl.parse(sheet, header=None)
    if (not raw.empty) and isinstance(raw.iat[0,0], str) and raw.iat[0,0].strip().lower() == "metadata":
        headers = raw.iloc[1].astype(str).tolist()
        df = raw.iloc[2:].copy()
        df.columns = headers
        df = df.reset_index(drop=True)
        return df
    # 他形式への保険：先頭5行をスキャン
    header_tokens = {"gene label","symbol","effect","average effect per pull-down pair","p value","p value (adjusted)"}
    for hi in range(min(5, len(raw))):
        row = raw.iloc[hi].astype(str).str.strip().str.lower().tolist()
        if any(tok in row for tok in header_tokens):
            headers = raw.iloc[hi].astype(str).tolist()
            df = raw.iloc[hi+1:].copy()
            df.columns = headers
            return df.reset_index(drop=True)
    # 既定
    return xl.parse(sheet, header=0)

# ---- 全タブを走査して、HDAC1/2/3の「平均効果（行単位）」を縦持ちで収集 ----
xl = pd.ExcelFile(wb_path)
sheets = xl.sheet_names  # 例：['DNA Methylation','H2A.Z','H3K27ac',...]
long_vals = []           # 各行（＝リプリケート相当）を1レコードに

for sheet in sheets:
    if sheet.lower().startswith("index"):
        continue
    df = read_protein_sheet(xl, sheet)

    # 列名（表記揺れに寛容）
    sym_col  = _find_col(df.columns, ["Symbol", "Gene symbol", "Gene", "Gene label"])
    eff_col  = _find_col(df.columns, ["Average effect per pull-down pair","Effect","Median effect","log2_effect","delta log2hl"])
    if sym_col is None or eff_col is None:
        # このタブでは必要列が見つからない → スキップ
        continue

    d = df[[sym_col, eff_col]].copy()
    d.columns = ["symbol", "effect"]
    d["symbol"] = d["symbol"].astype(str).str.upper()
    d["feature"] = sheet
    d = d[d["symbol"].isin(genes_of_interest)]
    d["effect"] = pd.to_numeric(d["effect"], errors="coerce")
    d = d.dropna(subset=["effect"])
    long_vals.append(d[["feature","symbol","effect"]])

vals_df = pd.concat(long_vals, ignore_index=True) if long_vals else pd.DataFrame(columns=["feature","symbol","effect"])

# ---- 各 feature × gene で中央値・四分位・min/max を集計（図の帯と髭に相当）----
summary = (vals_df
           .groupby(["feature","symbol"])["effect"]
           .agg(n="count",
                median="median",
                q1=lambda s: s.quantile(0.25),
                q3=lambda s: s.quantile(0.75),
                vmin="min",
                vmax="max")
           .reset_index())

# マークの表示順（ワークブックの順序を基本に、実際にデータがあるもののみ）
feature_order = [s for s in sheets if s in summary["feature"].unique()]
summary["feature"] = pd.Categorical(summary["feature"], categories=feature_order, ordered=True)
vals_df["feature"] = pd.Categorical(vals_df["feature"], categories=feature_order, ordered=True)

summary = summary.sort_values(["feature","symbol"]).reset_index(drop=True)

print("features (n) =", len(feature_order), "rows(summary) =", len(summary))
summary.head()


In [ ]:
# === 添付図風：各featureに対して HDAC1/2/3 の分布を水平表示（色分け） ===
import matplotlib.pyplot as plt
import matplotlib as mpl
from math import ceil

# 色（ユーザー要望により明示的に指定）
colors = {"HDAC1": "#1f77b4", "HDAC2": "#ff7f0e", "HDAC3": "#2ca02c"}  # 適宜変更可

# 図の縦サイズを feature 数から自動設定
h = 0.35 * len(summary["feature"].cat.categories) + 1.5
fig = plt.figure(figsize=(8, h))
ax = plt.gca()

# 中央線（x=0）
ax.axvline(0, lw=1, color="k")

# Y軸のベース位置（featureごとに1行）
feat_to_y = {feat: i for i, feat in enumerate(summary["feature"].cat.categories)}
# 同一feature内で HDAC1/2/3 を見分けるための微小オフセット
gene_offsets = {"HDAC1": -0.18, "HDAC2": 0.0, "HDAC3": 0.18}

# まずは各点（各行＝リプリケート相当の effect）を薄めの点で
for g in ["HDAC1","HDAC2","HDAC3"]:
    subp = vals_df[vals_df["symbol"]==g]
    for feat, grp in subp.groupby("feature"):
        y = feat_to_y[feat] + gene_offsets[g]
        ax.scatter(grp["effect"], [y]*len(grp), s=18, alpha=0.6, color=colors[g], linewidths=0, zorder=2)

# 次に summary（IQR帯とmin–max髭、中央値●）
for _, row in summary.iterrows():
    feat, g = row["feature"], row["symbol"]
    y = feat_to_y[feat] + gene_offsets.get(g, 0.0)
    # min–max の細い水平線（髭）
    ax.hlines(y, row["vmin"], row["vmax"], lw=1.2, color=colors[g], zorder=3)
    # IQR（Q1–Q3）の太線（帯）
    ax.hlines(y, row["q1"], row["q3"], lw=6, alpha=0.35, color=colors[g], zorder=4)
    # 中央値 ●
    ax.scatter([row["median"]], [y], s=40, color=colors[g], edgecolor="black", linewidths=0.5, zorder=5)

# 体裁：Y軸（feature名）、反転して上から順に表示
ax.set_yticks(range(len(feat_to_y)))
ax.set_yticklabels(summary["feature"].cat.categories.tolist())
ax.invert_yaxis()
ax.set_xlabel("PTM contribution to H/L ratio (log2)")
ax.set_title("Chromatin feature effect estimates — HDAC1/2/3")

# 凡例
for g in ["HDAC1","HDAC2","HDAC3"]:
    ax.scatter([], [], color=colors[g], label=g)
ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.show()
# 保存するなら：
# plt.savefig("results/figures/hdac123_feature_effects.png", dpi=200, bbox_inches="tight")


In [ ]:
# === H3ac / H3K4me3 / H4ac のみ・jitter除去・横補助線・x範囲[-0.5, 0.8] 固定版 ===
import matplotlib.pyplot as plt

# 対象 feature 絞り込み
target_features = ["H3ac", "H4ac", "H3K4me3"]
vals_df_sub = vals_df[vals_df["feature"].isin(target_features)].copy()
summary_sub = summary[summary["feature"].isin(target_features)].copy()

# カテゴリ順（指定順で存在するもののみ）
feature_order = [f for f in target_features if f in vals_df_sub["feature"].unique()]
for df_ in (vals_df_sub, summary_sub):
    df_["feature"] = pd.Categorical(df_["feature"], categories=feature_order, ordered=True)

# 再ソート
summary_sub = summary_sub.sort_values(["feature","symbol"]).reset_index(drop=True)

# x軸範囲
x_min, x_max = -0.5, 0.8

# 図高さ
h = 0.9 * len(feature_order) + 1.2
fig = plt.figure(figsize=(7.2, h))
ax = plt.gca()

colors = {"HDAC1": "#1f77b4", "HDAC2": "#ff7f0e", "HDAC3": "#2ca02c"}

# jitter除去（全て 0）
gene_offsets = {"HDAC1": 0.0, "HDAC2": 0.0, "HDAC3": 0.0}

# y 座標割当
feat_to_y = {feat: i for i, feat in enumerate(feature_order)}

# 背景の横補助線（各ヒストン修飾の行を視覚化）
for feat, y in feat_to_y.items():
    ax.hlines(y, x_min, x_max, color="#d0d0d0", lw=0.8, zorder=0)

# 中央線
ax.axvline(0, lw=1, color="k", zorder=1)

# 各リプリケート点（同一yに重ねる）
for g in ["HDAC1","HDAC2","HDAC3"]:
    subp = vals_df_sub[vals_df_sub["symbol"]==g]
    for feat, grp in subp.groupby("feature"):
        y = feat_to_y[feat] + gene_offsets[g]
        ax.scatter(grp["effect"], [y]*len(grp), s=26, alpha=0.65, color=colors[g], linewidths=0, zorder=3)

# 要約（min-max / IQR / median）
for _, row in summary_sub.iterrows():
    feat, g = row["feature"], row["symbol"]
    if feat not in feat_to_y: 
        continue
    y = feat_to_y[feat] + gene_offsets.get(g, 0.0)
    # 値を表示範囲にクリップ（外側に伸びる線を防止）
    vmin = max(x_min, row["vmin"])
    vmax = min(x_max, row["vmax"])
    q1 = max(x_min, row["q1"])
    q3 = min(x_max, row["q3"])
    med = min(max(row["median"], x_min), x_max)
    ax.hlines(y, vmin, vmax, lw=1.2, color=colors[g], zorder=4)          # whisker
    ax.hlines(y, q1, q3, lw=6, alpha=0.35, color=colors[g], zorder=5)    # IQR
    ax.scatter([med], [y], s=48, color=colors[g], edgecolor="black", linewidths=0.5, zorder=6)

# 体裁
ax.set_yticks([feat_to_y[f] for f in feature_order])
ax.set_yticklabels(feature_order)
ax.invert_yaxis()
ax.set_xlim(x_min, x_max)
ax.set_xlabel("PTM contribution to H/L ratio (log2)")
ax.set_title("Chromatin feature effect estimates — HDAC1/2/3")

# 凡例
for g in ["HDAC1","HDAC2","HDAC3"]:
    ax.scatter([], [], color=colors[g], label=g)
ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig(MARCS_OUTPUT_ROOT / "figures/hdac123_feature_effects_subset.png", dpi=200)
plt.savefig(MARCS_OUTPUT_ROOT / "figures/hdac123_feature_effects_subset.pdf")
plt.show()

In [ ]:
# === HDAC1/2/3: Proteins workbook から全タブ横断で効果分布を抽出・集計 ===
from pathlib import Path
import pandas as pd
import numpy as np

wb_path = MARCS_DATA_ROOT / "marcs.ptm-response-proteins.2022-03-22.824b7d4.xlsx"  # 必要なら修正
genes_of_interest = ["HDAC1", "HDAC2", "HDAC3", "NCOR1", "NCOR2", "SAP30", "SIN3A"]  # 色分け対象

# ---- ヘッダー検出ユーティリティ（proteins特有: 最初がMetadata＋次行が見出し） ----
def _norm(s):
    return "".join(ch for ch in str(s).strip().lower() if ch.isalnum())

def _find_col(cols, candidates):
    norm_map = {_norm(c): c for c in cols}
    # 完全一致優先
    for cand in candidates:
        key = _norm(cand)
        if key in norm_map:
            return norm_map[key]
    # 部分一致許容
    for c in cols:
        nc = _norm(c)
        if any(_norm(cand) in nc for cand in candidates):
            return c
    return None

def read_protein_sheet(xl, sheet):
    """Metadata行→次行が実ヘッダーの形式に対応。該当しない場合は通常のheader=0。"""
    raw = xl.parse(sheet, header=None)
    if (not raw.empty) and isinstance(raw.iat[0,0], str) and raw.iat[0,0].strip().lower() == "metadata":
        headers = raw.iloc[1].astype(str).tolist()
        df = raw.iloc[2:].copy()
        df.columns = headers
        df = df.reset_index(drop=True)
        return df
    # 他形式への保険：先頭5行をスキャン
    header_tokens = {"gene label","symbol","effect","average effect per pull-down pair","p value","p value (adjusted)"}
    for hi in range(min(5, len(raw))):
        row = raw.iloc[hi].astype(str).str.strip().str.lower().tolist()
        if any(tok in row for tok in header_tokens):
            headers = raw.iloc[hi].astype(str).tolist()
            df = raw.iloc[hi+1:].copy()
            df.columns = headers
            return df.reset_index(drop=True)
    # 既定
    return xl.parse(sheet, header=0)

# ---- 全タブを走査して、HDAC1/2/3の「平均効果（行単位）」を縦持ちで収集 ----
xl = pd.ExcelFile(wb_path)
sheets = xl.sheet_names  # 例：['DNA Methylation','H2A.Z','H3K27ac',...]
long_vals = []           # 各行（＝リプリケート相当）を1レコードに

for sheet in sheets:
    if sheet.lower().startswith("index"):
        continue
    df = read_protein_sheet(xl, sheet)

    # 列名（表記揺れに寛容）
    sym_col  = _find_col(df.columns, ["Symbol", "Gene symbol", "Gene", "Gene label"])
    eff_col  = _find_col(df.columns, ["Average effect per pull-down pair","Effect","Median effect","log2_effect","delta log2hl"])
    if sym_col is None or eff_col is None:
        # このタブでは必要列が見つからない → スキップ
        continue

    d = df[[sym_col, eff_col]].copy()
    d.columns = ["symbol", "effect"]
    d["symbol"] = d["symbol"].astype(str).str.upper()
    d["feature"] = sheet
    d = d[d["symbol"].isin(genes_of_interest)]
    d["effect"] = pd.to_numeric(d["effect"], errors="coerce")
    d = d.dropna(subset=["effect"])
    long_vals.append(d[["feature","symbol","effect"]])

vals_df = pd.concat(long_vals, ignore_index=True) if long_vals else pd.DataFrame(columns=["feature","symbol","effect"])

# ---- 各 feature × gene で中央値・四分位・min/max を集計（図の帯と髭に相当）----
summary = (vals_df
           .groupby(["feature","symbol"])["effect"]
           .agg(n="count",
                median="median",
                q1=lambda s: s.quantile(0.25),
                q3=lambda s: s.quantile(0.75),
                vmin="min",
                vmax="max")
           .reset_index())

# マークの表示順（ワークブックの順序を基本に、実際にデータがあるもののみ）
feature_order = [s for s in sheets if s in summary["feature"].unique()]
summary["feature"] = pd.Categorical(summary["feature"], categories=feature_order, ordered=True)
vals_df["feature"] = pd.Categorical(vals_df["feature"], categories=feature_order, ordered=True)

In [ ]:
# === HDAC1/2/3, NCOR1/2, SAP30, SIN3A の 3特徴（H3ac/H4ac/H3K4me3）図：上部凡例＋青点線(y=0.5)＋縦長 ===
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 対象遺伝子（色分け）
genes_of_interest = ["HDAC1", "HDAC2", "HDAC3", "NCOR1", "NCOR2", "SAP30", "SIN3A"]
colors = {
    "HDAC1": "#1f77b4",  # 青
    "HDAC2": "#ff7f0e",
    "HDAC3": "#2ca02c",
    "NCOR1": "#d62728",
    "NCOR2": "#9467bd",
    "SAP30": "#8c564b",
    "SIN3A": "#e377c2",
}

# 3特徴のみを使用
target_features = ["H3ac", "H4ac", "H3K4me3"]
vals_df_sub = vals_df[vals_df["feature"].isin(target_features)].copy()
summary_sub = summary[summary["feature"].isin(target_features)].copy()

# カテゴリ順（指定順に固定）
feature_order = [f for f in target_features if f in vals_df_sub["feature"].unique()]
for df_ in (vals_df_sub, summary_sub):
    df_["feature"] = pd.Categorical(df_["feature"], categories=feature_order, ordered=True)
summary_sub = summary_sub.sort_values(["feature","symbol"]).reset_index(drop=True)

# 軸範囲（先の設定を流用／任意に変更可）
y_min, y_max = -0.5, 1.7   # 例：先の x_min/x_max と同じレンジ
# 図サイズ（縦長に調整）
w = 0.5 * len(feature_order) + 1.2
h = 9                     # ← 少し縦長に
fig = plt.figure(figsize=(w, h))
ax = plt.gca()

# 縦の補助線（各特徴の列）
feat_to_x = {feat: i for i, feat in enumerate(feature_order)}
for feat, x0 in feat_to_x.items():
    ax.vlines(x0, y_min, y_max, color="#d0d0d0", lw=0.8, zorder=0)

# y=0 と y=0.5（青点線）の基準線
ax.axhline(0.0, lw=1, color="k", zorder=1)
ax.axhline(0.5, lw=1, ls=":", color="tab:blue", zorder=1)  # ★ 追加：青い点線

# 各リプリケート点（同一xに重ねる；ジッタ無し）
for g in genes_of_interest:
    subp = vals_df_sub[vals_df_sub["symbol"] == g]
    if subp.empty:
        continue
    for feat, grp in subp.groupby("feature"):
        x0 = feat_to_x[feat]
        ax.scatter([x0]*len(grp), grp["effect"], s=26, alpha=0.65, color=colors[g], linewidths=0, zorder=3)

# 要約（min–max / IQR / median）を縦方向に描画
for _, row in summary_sub.iterrows():
    feat, g = row["feature"], row["symbol"]
    if feat not in feat_to_x or g not in colors:
        continue
    x0 = feat_to_x[feat]
    vmin = max(y_min, row["vmin"])
    vmax = min(y_max, row["vmax"])
    q1 = max(y_min, row["q1"])
    q3 = min(y_max, row["q3"])
    med = min(max(row["median"], y_min), y_max)
    ax.vlines(x0, vmin, vmax, lw=1.2, color=colors[g], zorder=4)       # whisker
    ax.vlines(x0, q1, q3, lw=6, alpha=0.35, color=colors[g], zorder=5) # IQR帯
    ax.scatter([x0], [med], s=48, color=colors[g], edgecolor="black", linewidths=0.5, zorder=6)

# 軸・ラベル
ax.set_xticks([feat_to_x[f] for f in feature_order])
ax.set_xticklabels(
    feature_order,
    rotation=-90,
    ha='left',
    va='top',               # 上端を基準に揃える
    rotation_mode='anchor'
)
ax.tick_params(axis='x', which='major', pad=5)  # ラベルを軸の外側へオフセット
ax.set_xlim(-0.5, len(feature_order) - 0.5)
ax.set_ylim(y_min, y_max)
ax.set_xlabel("Chromatin feature")
ax.set_ylabel("PTM contribution to H/L ratio (log2)")

# ==== 凡例（上部） ====
legend_ncol = 2
handles = [Line2D([], [], marker='o', linestyle='None', color=colors[g], label=g)
           for g in genes_of_interest]
leg = fig.legend(
    handles=handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.985),  # ほんの少し上へ
    ncol=legend_ncol,
    frameon=False,
    handletextpad=0.4,
    columnspacing=1.0,
    borderaxespad=0.0
)

# 上部・下部マージンを調整して、凡例との空きとxラベル用の余白を最適化
plt.tight_layout(rect=(0.0, 0.0, 1.0, 0.91))
# ...existing code...
plt.savefig(MARCS_OUTPUT_ROOT / "figures/selected_genes_feature_effects_subset_rotated_toplegend.png", dpi=200)
plt.savefig(MARCS_OUTPUT_ROOT / "figures/selected_genes_feature_effects_subset_rotated_toplegend.pdf")
plt.show()


In [ ]:
# === HDAC1/2/3, NCOR1/2, SAP30, SIN3A の 3特徴図：x軸を上側に、凡例は下、図サイズ 6x4 ===
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

genes_of_interest = ["HDAC1", "HDAC2", "HDAC3", "NCOR1", "NCOR2", "SAP30", "SIN3A"]
colors = {
    "HDAC1": "#1f77b4", "HDAC2": "#ff7f0e", "HDAC3": "#2ca02c",
    "NCOR1": "#d62728", "NCOR2": "#9467bd", "SAP30": "#8c564b", "SIN3A": "#e377c2",
}

target_features = ["H3ac", "H4ac", "H3K4me3"]
vals_df_sub = vals_df[vals_df["feature"].isin(target_features)].copy()
summary_sub  = summary[summary["feature"].isin(target_features)].copy()

feature_order = [f for f in target_features if f in vals_df_sub["feature"].unique()]
for df_ in (vals_df_sub, summary_sub):
    df_["feature"] = pd.Categorical(df_["feature"], categories=feature_order, ordered=True)
summary_sub = summary_sub.sort_values(["feature","symbol"]).reset_index(drop=True)

x_min, x_max = -0.5, 1.7


fig = plt.figure(figsize=(6.0, 4.0))
ax  = plt.gca()



feat_to_y = {feat: i for i, feat in enumerate(feature_order)}
for feat, y0 in feat_to_y.items():
    ax.hlines(y0, x_min, x_max, color="#d0d0d0", lw=0.8, zorder=0)

ax.axvline(0.0, lw=1, color="k", zorder=1)
ax.axvline(0.5, lw=1, ls=":", color="tab:blue", zorder=1)

for g in genes_of_interest:
    subp = vals_df_sub[vals_df_sub["symbol"] == g]
    if subp.empty: 
        continue
    for feat, grp in subp.groupby("feature"):
        y0 = feat_to_y[feat]
        ax.scatter(grp["effect"], [y0]*len(grp), s=26, alpha=0.65,
                   color=colors.get(g, "0.3"), linewidths=0, zorder=3)

for _, row in summary_sub.iterrows():
    feat, g = row["feature"], row["symbol"]
    if feat not in feat_to_y or g not in colors: 
        continue
    y0   = feat_to_y[feat]
    vmin = max(x_min, row["vmin"]); vmax = min(x_max, row["vmax"])
    q1   = max(x_min, row["q1"]);   q3   = min(x_max, row["q3"])
    med  = min(max(row["median"], x_min), x_max)
    ax.hlines(y0, vmin, vmax, lw=1.2, color=colors[g], zorder=4)        # whisker
    ax.hlines(y0, q1, q3,   lw=6,   alpha=0.35, color=colors[g], zorder=5) # IQR
    ax.scatter([med], [y0], s=48, color=colors[g], edgecolor="black", linewidths=0.5, zorder=6)

# --- 軸・ラベル：x軸を上側に移動 ---
ax.set_yticks([feat_to_y[f] for f in feature_order])
ax.set_yticklabels(feature_order)
ax.invert_yaxis()
ax.set_xlim(x_min, x_max)

ax.set_xlabel("PTM contribution to H/L ratio (log2)", labelpad=6)
ax.set_ylabel("Chromatin feature")

# 目盛・ラベルを上に、下は消す
ax.tick_params(axis='x', bottom=False, top=True, labelbottom=False, labeltop=True)
ax.xaxis.set_label_position('top')

# 凡例（下中央、図幅内）
handles = [Line2D([], [], marker='o', linestyle='None', color=colors[g], label=g)
           for g in genes_of_interest]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.62, 0.02),
           ncol=4, frameon=False, handletextpad=0.4, columnspacing=1.0, borderaxespad=0.0)

# 余白調整：上に軸ラベルを置くので top を少し下げ、下は凡例用のスペースを確保
plt.tight_layout(rect=(0.02, 0.14, 1.0, 0.90))

plt.savefig(MARCS_OUTPUT_ROOT / "figures/selected_genes_feature_effects_subset_rotated_right_topx.png", dpi=200)
plt.savefig(MARCS_OUTPUT_ROOT / "figures/selected_genes_feature_effects_subset_rotated_right_topx.pdf")
plt.show()


In [ ]:
# === 3特徴図：中央値のみを縦方向ジッター表示（重なり回避） ===
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

genes_of_interest = ["HDAC1","HDAC2","HDAC3","NCOR1","NCOR2","SAP30","SIN3A"]
colors = {
    "HDAC1":"#1f77b4","HDAC2":"#ff7f0e","HDAC3":"#2ca02c",
    "NCOR1":"#d62728","NCOR2":"#9467bd","SAP30":"#8c564b","SIN3A":"#e377c2",
}

target_features = ["H3ac","H4ac","H3K4me3"]
vals_df_sub = vals_df[vals_df["feature"].isin(target_features)].copy()
summary_sub  = summary[summary["feature"].isin(target_features)].copy()

feature_order = [f for f in target_features if f in vals_df_sub["feature"].unique()]
for df_ in (vals_df_sub, summary_sub):
    df_["feature"] = pd.Categorical(df_["feature"], categories=feature_order, ordered=True)
summary_sub = summary_sub.sort_values(["feature","symbol"]).reset_index(drop=True)

x_min, x_max = -0.5, 1.7

# ジッター設定
jitter_y = 0.05            # 縦方向ジッター幅（カテゴリ間隔=1の座標系）
rng_seed = 42              # 再現性用（不要なら None）
jitter_mode = "random"     # "random" か "dodge"
# dodge（固定オフセット）を使う場合：遺伝子数に応じて手動定義
dodge_offsets = {
    # 均等配置（幅 ≈ jitter_y）: 中央を 0 にして左右へ
    "HDAC1": -0.24, "HDAC2": -0.12, "HDAC3": 0.00,
    "NCOR1": 0.12,  "NCOR2": 0.24,  "SAP30": 0.36, "SIN3A": -0.36
}

if rng_seed is not None:
    rng = np.random.default_rng(rng_seed)
else:
    rng = np.random.default_rng()

# 中央値のみ抽出
medians_df = summary_sub[["feature","symbol","median"]].copy()

fig = plt.figure(figsize=(6.0, 4.0))
ax  = plt.gca()

feat_to_y = {feat:i for i,feat in enumerate(feature_order)}
for feat, y0 in feat_to_y.items():
    ax.hlines(y0, x_min, x_max, color="#d0d0d0", lw=0.8, zorder=0)

ax.axvline(0.0, lw=1, color="k", zorder=1)
ax.axvline(0.5, lw=1, ls=":", color="tab:blue", zorder=1)

# 中央値点（ジッター適用）
y_positions = []
for _, row in medians_df.iterrows():
    feat, g, med = row["feature"], row["symbol"], row["median"]
    if feat not in feat_to_y or g not in colors:
        continue
    base_y = feat_to_y[feat]
    if jitter_mode == "random":
        y_plot = base_y + rng.uniform(-jitter_y, jitter_y)
    else:  # dodge
        y_plot = base_y + dodge_offsets.get(g, 0.0)
    y_positions.append(y_plot)
    ax.scatter(med, y_plot, s=60, color=colors[g], edgecolor="black",
               linewidths=0.5, zorder=5)

# 重なり確認用（任意）: print(list(zip(medians_df["symbol"], y_positions)))

# 体裁
ax.set_yticks([feat_to_y[f] for f in feature_order])
ax.set_yticklabels(feature_order)
ax.invert_yaxis()
ax.set_xlim(x_min, x_max)
ax.set_xlabel("PTM contribution to H/L ratio (log2)", labelpad=6)
ax.set_ylabel("Chromatin feature")
ax.tick_params(axis='x', bottom=False, top=True, labelbottom=False, labeltop=True)
ax.xaxis.set_label_position('top')

handles = [Line2D([], [], marker='o', linestyle='None', color=colors[g], label=g)
           for g in genes_of_interest]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.62, 0.02),
           ncol=4, frameon=False, handletextpad=0.4, columnspacing=1.0, borderaxespad=0.0)

plt.tight_layout(rect=(0.02, 0.14, 1.0, 0.90))
plt.savefig(MARCS_OUTPUT_ROOT / "figures/selected_genes_feature_effects_subset_rotated_right_topx.png", dpi=200)
plt.savefig(MARCS_OUTPUT_ROOT / "figures/selected_genes_feature_effects_subset_rotated_right_topx.pdf")
plt.show()